In [ ]:
import sys
import os

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules
if(IN_COLAB):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    username = "ptowncs"
    git_repo = "super_gans"

    !git clone https://{token}@github.com/{username}/{git_repo}.git

    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/{git_repo}



# ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config as cfg

def create_dirs():
    dirs = [cfg.DATA_DIR, cfg.RESULTS_DIR, cfg.MODELS_DIR]
    for d in dirs:
        os.makedirs(d, exist_ok=True)
create_dirs()


In [ ]:
#from tensorflow.keras.models import load_model

# Load the custom generator from GAN A
#generator_a = load_model("/kaggle/input/<your-uploaded-folder>//kaggle/working/srgan.h5", compile=False)


In [ ]:
# #generate high quality images
# from tensorflow.keras.preprocessing.image import load_img, img_to_array, array_to_img
# import numpy as np

# # Load a low-res image (ensure this matches your generator's input size, e.g., 64x64)
# lr_img_path = "/kaggle/input/chest-xray-pneumonia/chest_xray/test/PNEUMONIA/some_image.jpeg"
# lr_img = load_img(lr_img_path, target_size=(64, 64))  # match generator input size
# lr_img_array = img_to_array(lr_img) / 255.0  # normalize image
# lr_img_array = np.expand_dims(lr_img_array, axis=0)  # add batch dimension

# # Generate high-res image using GAN A's generator
# sr_img_array = generator_a.predict(lr_img_array)

# # Convert the generated high-res image back to an image and save it
# sr_img = array_to_img(sr_img_array[0])
# sr_img.save("generated_high_res_image.png")


In [ ]:
dataset_path = f"{cfg.DATA_DIR}/datasets/paultimothymooney/chest-xray-pneumonia"
chest_xray_ds = f"{dataset_path}/versions/2/chest_xray"

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter  # to print to tensorboard
from torch.utils.data import RandomSampler

class Discriminator(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(in_features, 128),# Analyzes image features
            nn.LeakyReLU(0.01), # Prevents dead neurons
            nn.Linear(128, 1), # Outputs a single number
            nn.Sigmoid(), # Converts into probability score
        )

    def forward(self, x):
        return self.disc(x) #Defines how data flows through the network


class Generator(nn.Module):
    def __init__(self, z_dim, img_dim): # z_dim is size of noise vector , img_dim is total pixels in image
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.01),
            nn.Linear(256, img_dim),
            nn.Tanh(),  # normalize inputs to [-1, 1] so make outputs [-1, 1]
        )

    def forward(self, x):
        return self.gen(x)


# Hyperparameters etc.
device = "cuda" if torch.cuda.is_available() else "cpu"
lr = 2e-4
z_dim = 64
image_size = 64
num_channels = 1
image_dim = image_size * image_size * num_channels
batch_size = 64
num_epochs = 400

disc = Discriminator(image_dim).to(device)
gen = Generator(z_dim, image_dim).to(device)
fixed_noise = torch.randn((batch_size, z_dim)).to(device)
transforms_pipeline = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels = num_channels),# Force 1 channel
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(), # image to tensor
        transforms.Normalize((0.5,), (0.5,)), # normalize images , [0,1] to [-1,1]
    ]
)


def is_valid_image(filename):
    return not filename.startswith("._")  # skip hidden macOS files

dataset = datasets.ImageFolder(
    root=f"{chest_xray_ds}/train"
,
    transform=transforms_pipeline,
    is_valid_file=is_valid_image
)




In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision


# DataLoader with full dataset and shuffling
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

opt_disc = optim.Adam(disc.parameters(), lr=lr)
opt_gen = optim.Adam(gen.parameters(), lr=lr)
criterion = nn.BCELoss()

writer_fake = SummaryWriter("logs/fake")
writer_real = SummaryWriter("logs/real")

step = 0

for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, image_dim).to(device) #Flatten image into vectors
        batch_size = real.shape[0]

        # Train Discriminator
        noise = torch.randn(batch_size, z_dim).to(device) # generator makes fakes
        fake = gen(noise)

        disc_real = disc(real).view(-1) # disc tries to classify real images
        lossD_real = criterion(disc_real, torch.ones_like(disc_real))

        disc_fake = disc(fake.detach()).view(-1) #disc tries to classify real images
        lossD_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

        lossD = (lossD_real + lossD_fake) / 2

        #Backpropagation
        disc.zero_grad()
        lossD.backward()
        opt_disc.step()

        # Train Generator
        output = disc(fake).view(-1) # Generator looks at images that were detected as fake
        lossG = criterion(output, torch.ones_like(output))

        # update generator weights
        gen.zero_grad()
        lossG.backward()
        opt_gen.step()

        if batch_idx == 0:
            print(f"Epoch [{epoch}/{num_epochs}] Batch {batch_idx}/{len(loader)} "
                  f"Loss D: {lossD.item():.4f}, loss G: {lossG.item():.4f}")

            with torch.no_grad():
                fake = gen(fixed_noise).reshape(-1, num_channels, image_size, image_size)
                data = real.reshape(-1, num_channels, image_size, image_size)

                img_grid_fake = torchvision.utils.make_grid(fake, normalize=True)
                img_grid_real = torchvision.utils.make_grid(data, normalize=True)

                writer_fake.add_image("Pneumonia Fake Images", img_grid_fake, global_step=step)
                writer_real.add_image("Pneumonia Real Images", img_grid_real, global_step=step)

                step += 1

In [ ]:
# Saving checkpoints
gan_checkpoints_dir = f"{cfg.MODELS_DIR}/gan_checkpoints"
gan_results_dir = f"{cfg.RESULTS_DIR}/generated_images_fid"
os.makedirs(gan_checkpoints_dir, exist_ok=True)
os.makedirs(gan_results_dir, exist_ok=True)

checkpoint = {
    "generator_state_dict": gen.state_dict(),
    "discriminator_state_dict": disc.state_dict(),
    "optimizer_G_state_dict": opt_gen.state_dict(),
    "optimizer_D_state_dict": opt_disc.state_dict(),
}

print("Saving GAN state to GAN Checkpoints")
torch.save(checkpoint, f"{gan_checkpoints_dir}/simple_gan_checkpoint.pth")


In [ ]:
import torchvision
from torchvision import transforms
import os
from PIL import Image # Import PIL Image

# Directory to save the real images for FID calculation
import torchvision
from torchvision import transforms
import os
from PIL import Image

# Directory to save the real images for FID calculation

real_images_dir = f"{cfg.RESULTS_DIR}/real_images_fid"
os.makedirs(real_images_dir, exist_ok=True)

num_real_images_to_save = 10000

for i in range(min(num_real_images_to_save, len(dataset))):
    image, _ = dataset[i]   # image is a tensor in shape (1, 64, 64)

    # Convert grayscale -> RGB by repeating channels
    image_rgb = image.repeat(3, 1, 1)


    torchvision.utils.save_image(
        image_rgb,
        os.path.join(real_images_dir, f"pneumonia_{i:04d}.png"),
        normalize=False,
        value_range=(-1, 1)
    )

print(f"Saved {min(num_real_images_to_save, len(dataset))} real Pneumonia images to {real_images_dir}/")


In [ ]:
#  Instantiate models
generator = Generator(z_dim, image_dim).to(device)
discriminator = Discriminator(image_dim).to(device) # Discriminator isn't strictly needed for FID, but useful if you need to inspect it.

#  Path to the checkpoint file
checkpoint_path = f"{gan_checkpoints_dir}/simple_gan_checkpoint.pth"

#  Load the checkpoint
try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    generator.load_state_dict(checkpoint["generator_state_dict"])
    # discriminator.load_state_dict(checkpoint["discriminator_state_dict"]) # Optional
    print(f"Checkpoint loaded from {checkpoint_path}")
except FileNotFoundError:
    print(f"Error: Checkpoint file not found at {checkpoint_path}")
    # Handle the error, maybe exit or use default models
except KeyError as e:
    print(f"Error loading state_dict: Missing key {e}. Ensure checkpoint structure matches.")
    # Handle error if checkpoint dictionary keys don't match.

#  Set generator to evaluation mode
generator.eval()

In [ ]:
from torchvision.utils import save_image
import os

# Generate random noise vectors
num_images_to_generate = 10000
gen_noise = torch.randn(num_images_to_generate, z_dim).to(device)

# Generate images
with torch.no_grad():
    generated_images = generator(gen_noise).reshape(-1, num_channels, image_size, image_size)# Reshape for saving/display

generated_images_dir = f"{cfg.RESULTS_DIR}/fake_images_fid"
os.makedirs(generated_images_dir, exist_ok=True)

# Iterate through the batch and save each image separately
for i, image in enumerate(generated_images):
    # Construct the filename for each image
    filename = os.path.join(generated_images_dir, f"generated_image_{i:04d}.png") # Using f-strings for formatted filename

    # Save the individual image (image tensor has shape (channels, height, width))
    save_image(image, filename, normalize=False)

print(f"{num_images_to_generate} individual images saved to {generated_images_dir}/")


In [ ]:
!pip install pytorch-fid

In [ ]:
import os
import torch
from pytorch_fid import fid_score

# Paths to your image directories
print(f"Path being checked:{repr(real_images_dir)}")
if not os.path.exists(real_images_dir):
    raise ValueError(f"Real image path not found: {real_images_dir}")
if not os.path.exists(generated_images_dir):
    raise ValueError(f"Generated image path not found: {generated_images_dir}")

# (Make sure to populate `real_images` with your actual dataset)
device = "cuda" if torch.cuda.is_available() else "cpu"
# Calculate FID
fid_value = fid_score.calculate_fid_given_paths(
    [real_images_dir, generated_images_dir],
    batch_size=50, # Adjust batch size based on available GPU memory
    device=device,
    dims=2048 # Inception v3 output dimension
)

print(f"FID score: {fid_value}")